In [ ]:
import os
import face_recognition
import pickle
import shutil
import cv2
import numpy as np
from PIL import Image


def process_images(directory):
    face_data = {"encodings": [], "filenames": [], "locations": []}
    for filename in os.listdir(directory):
        if filename.endswith(('.jpg', '.png')):
            image_path = os.path.join(directory, filename)
            image = face_recognition.load_image_file(image_path)
            face_locations = face_recognition.face_locations(image)
            face_encodings = face_recognition.face_encodings(image, face_locations)
            for encoding, location in zip(face_encodings, face_locations):
                face_data["encodings"].append(encoding)
                face_data["locations"].append(location)
                face_data["filenames"].append(filename)

    with open("stored_face_data.pkl", "wb") as f:
        pickle.dump(face_data, f)

def search_face(target_face_encoding):
    with open("stored_face_data.pkl", "rb") as f:
        stored_face_data = pickle.load(f)
    matching_faces = []
    for i, stored_encoding in enumerate(stored_face_data["encodings"]):
        if face_recognition.compare_faces([stored_encoding], target_face_encoding)[0]:
            matching_faces.append(stored_face_data["filenames"][i])
    return matching_faces

def detect_and_search_faces(image_path):
    image = face_recognition.load_image_file(image_path)
    face_locations = face_recognition.face_locations(image)
    face_encodings = face_recognition.face_encodings(image, face_locations)
    matching_photos = []
    for i, face_encoding in enumerate(face_encodings):
        matching_photos.extend(search_face(face_encoding))
    return image, matching_photos

def remove_duplicates_from_dict_values(dictionary):
    for key, value in dictionary.items():
        # Convert the list of values to a set to remove duplicates
        unique_values = set(value)
        # Convert the set back to a list
        dictionary[key] = list(unique_values)
    return dictionary

#############33encryption##########
def encrypt_image_from_file(file_name, output_folder):
    try:
        # Load the image
        img = Image.open(file_name)
        img_array = np.array(img)

        encrypted_images = [np.zeros_like(img_array) for _ in range(3)]

        for i in range(img_array.shape[0]):
            for j in range(img_array.shape[1]):
                # If it's within the first 500 pixels, copy pixel values directly
                if i * img_array.shape[1] + j < 500:
                    for k in range(3):
                        encrypted_images[k][i, j] = img_array[i, j]
                else:
                    # Otherwise, compute residues modulo 3, 5, and 17 for each pixel
                    encrypted_images[0][i, j] = img_array[i, j] % 3
                    encrypted_images[1][i, j] = img_array[i, j] % 5
                    encrypted_images[2][i, j] = img_array[i, j] % 17

        # Create the output folder if it doesn't exist
        os.makedirs(output_folder, exist_ok=True)

        # Save the encrypted images
        for k in range(3):
            encrypted_img = Image.fromarray(encrypted_images[k])
            encrypted_img.save(os.path.join(output_folder, f"encrypted_{k}.png"))

        print("Encryption completed successfully!")
    except Exception as e:
        print(f"Error during encryption: {e}")

training_folder_path = r"C:\Users\thash\OneDrive\Desktop\test\for image processing Train"

process_images(training_folder_path)

with open("stored_face_data.pkl", "rb") as f:
    stored_face_data = pickle.load(f)

folder_path = "ptest"
output_file = open("output.txt", "w")
images_with_persons = {}

# Process the testing images
for filename in os.listdir(folder_path):
    if filename.endswith(('.jpg', '.jpeg', '.png')):
        target_image_path = os.path.join(folder_path, filename)
        image, matching_faces = detect_and_search_faces(target_image_path)
        if matching_faces:
            images_with_persons[filename] = matching_faces
        output_file.write("Faces detected in " + filename + ":\n")
        for person in matching_faces:
            output_file.write("- Person: " + person + "\n")
        output_file.write("\n")
output_file.close()

images_with_persons = remove_duplicates_from_dict_values(images_with_persons)
print("Images with detected persons:", images_with_persons)

for key in images_with_persons.keys():
    filename = os.path.join(folder_path, key)
    if os.path.isfile(filename):
        input_image_file = filename
        
        subfolder_name = os.path.splitext(key)[0]
        
        output_subdirectory = os.path.join("output_folder", subfolder_name)
        os.makedirs(output_subdirectory, exist_ok=True)
        
        #########encrypt 
        output_folder_path = output_subdirectory
        encrypt_image_from_file(input_image_file, output_folder_path)Images with detected persons: {'Akshay-Kumar-with-a-fan.jpg'

In [ ]:
import os
from PIL import Image
import numpy as np

def multiplicative_inverse(a, m):
    m0, x0, x1 = m, 0, 1
    if m == 1:
        return 0
    while a > 1:
        q = a // m
        m, a = a % m, m
        x0, x1 = x1 - q * x0, x0
    return x1 + m0 if x1 < 0 else x1

def crt(pixels, moduli):
    total = 0
    prod = np.prod(moduli, dtype=np.uint64)
    for pixel, modulus in zip(pixels, moduli):
        p = prod // modulus
        inv = multiplicative_inverse(p, modulus)
        total += pixel * inv * p
    return total % prod

def decrypt_image(encrypted_paths, moduli):
    encrypted_images = [np.array(Image.open(path), dtype=np.uint64) for path in encrypted_paths]

    decrypted_img_array = np.zeros_like(encrypted_images[0])

    for i in range(decrypted_img_array.shape[0]):
        for j in range(decrypted_img_array.shape[1]):
            pixels = [img[i, j] for img in encrypted_images]
            decrypted_img_array[i, j] = crt(pixels, moduli)

    decrypted_img_array = decrypted_img_array.astype(np.uint8)

    return Image.fromarray(decrypted_img_array)

def decrypt_folder_for_filename(folder_path, filename):
    folder_path = os.path.join('output_folder', filename)
    encrypted_paths = [os.path.join(folder_path, f'encrypted_{i}.png') for i in range(3)]
    moduli = [3, 5, 17]
    decrypted_img = decrypt_image(encrypted_paths, moduli)
    decrypted_img.save(os.path.join(folder_path, 'decrypted_image.png'))

filename = input("Enter the folder name inside the folder encrypted_inmages: ")

decrypt_folder_for_filename('encrypted_images', filename)